# YOLO26 Model Comparison Study

Trains all five YOLO26 detection variants on the DeepSense Scenario 23 drone dataset (100 epochs each) and compares accuracy, training time, and inference speed.

**Setup:** Runtime → Change runtime type → **T4 GPU** (or better).

| Model | Params | GFLOPs (640) | COCO mAP50-95 |
| --- | --- | --- | --- |
| YOLO26n (Nano) | 2.4M | 5.4 | 40.9 |
| YOLO26s (Small) | 9.5M | 20.7 | 48.6 |
| YOLO26m (Medium) | 20.4M | 68.2 | 53.1 |
| YOLO26l (Large) | 24.8M | 86.4 | 55.0 |
| YOLO26x (XLarge) | 55.7M | 193.9 | 57.5 |

In [ ]:
# Install dependencies (kernel may restart — this is normal, just continue to the next cell)
%pip install -q ultralytics matplotlib

In [ ]:
# Clone repo and cd into it
import os

if not os.path.exists("beamfinder"):
    !git clone https://github.com/seriouslysahid/beamfinder.git
else:
    !cd beamfinder && git pull

%cd beamfinder

In [ ]:
# Verify GPU
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Train all YOLO26 variants (with crash recovery)
import time, json, shutil
import torch
from pathlib import Path
from ultralytics import YOLO

MODELS = ["yolo26n.pt", "yolo26s.pt", "yolo26m.pt", "yolo26l.pt", "yolo26x.pt"]
RESULTS_FILE = Path("runs/study/results_summary.json")
RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)

# Resume from previous results if kernel restarted
if RESULTS_FILE.exists():
    results = json.loads(RESULTS_FILE.read_text())
    done = {r["model"] for r in results}
    print(f"Resuming \u2014 {len(done)} models already done: {done}")
else:
    results = []
    done = set()

TRAIN_ARGS = dict(
    data="data.yaml", epochs=100, imgsz=960, batch=0.85,
    patience=20, cache="disk", workers=4, cos_lr=True,
    deterministic=False, rect=True, save_period=10,
    degrees=15.0, flipud=0.5, scale=0.9, translate=0.2,
)

for i, pt_file in enumerate(MODELS):
    name = Path(pt_file).stem
    run_dir = Path(f"runs/study/{name}")

    if name in done:
        print(f"\n[{i+1}/{len(MODELS)}] {name} \u2014 SKIPPED (already done)")
        continue

    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(MODELS)}] Training {name}")
    print(f"{'='*60}\n")

    model = YOLO(pt_file)
    params = sum(p.numel() for p in model.model.parameters()) / 1e6

    t0 = time.time()
    model.train(**TRAIN_ARGS, project="runs/study", name=name, exist_ok=True)
    train_time = time.time() - t0

    # Load best weights for evaluation
    best_pt = run_dir / "weights" / "best.pt"
    if best_pt.exists():
        model = YOLO(str(best_pt))

    val = model.val(data="data.yaml", imgsz=960)
    test = model.val(data="data.yaml", split="test", imgsz=960)

    results.append({
        "model": name,
        "params_M": round(params, 1),
        "train_time_min": round(train_time / 60, 1),
        "val_mAP50": round(val.box.map50, 4),
        "val_mAP50_95": round(val.box.map, 4),
        "test_mAP50": round(test.box.map50, 4),
        "test_mAP50_95": round(test.box.map, 4),
        "inference_ms": round(val.speed["inference"], 2),
    })
    done.add(name)

    # Save results after each model (crash recovery)
    RESULTS_FILE.write_text(json.dumps(results, indent=2))
    print(f"\n\u2713 {name} done \u2014 saved to {RESULTS_FILE}")

    del model
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"All {len(MODELS)} models completed!")
print(f"{'='*60}")

In [ ]:
# Summary table
import pandas as pd
import json
from pathlib import Path

results = json.loads(Path("runs/study/results_summary.json").read_text())
df = pd.DataFrame(results)
df.columns = ["Model", "Params (M)", "Train Time (min)",
              "Val mAP50", "Val mAP50-95", "Test mAP50", "Test mAP50-95",
              "Inference (ms)"]
df.style.highlight_max(
    subset=["Val mAP50", "Val mAP50-95", "Test mAP50", "Test mAP50-95"],
    color="#c6efce"
).highlight_min(
    subset=["Train Time (min)", "Inference (ms)"],
    color="#c6efce"
)

In [ ]:
# Comparison bar charts
import matplotlib.pyplot as plt
import json
from pathlib import Path

results = json.loads(Path("runs/study/results_summary.json").read_text())
labels = [r["model"].replace("yolo26", "").upper() for r in results]
colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#DDA0DD"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("YOLO26 Model Comparison \u2014 Drone Detection (100 Epochs, imgsz=960)",
             fontsize=15, fontweight="bold")

x = range(len(results))
w = 0.35

# ---- mAP@50 ----
ax = axes[0, 0]
v50 = [r["val_mAP50"] for r in results]
t50 = [r["test_mAP50"] for r in results]
b1 = ax.bar([i - w/2 for i in x], v50, w, label="Val", color="#4CAF50", edgecolor="white")
b2 = ax.bar([i + w/2 for i in x], t50, w, label="Test", color="#2196F3", edgecolor="white")
ax.set_title("mAP@50", fontweight="bold")
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.legend(); ax.set_ylim(0, 1)
ax.bar_label(b1, fmt="%.3f", fontsize=7, padding=2)
ax.bar_label(b2, fmt="%.3f", fontsize=7, padding=2)

# ---- mAP@50-95 ----
ax = axes[0, 1]
v95 = [r["val_mAP50_95"] for r in results]
t95 = [r["test_mAP50_95"] for r in results]
b1 = ax.bar([i - w/2 for i in x], v95, w, label="Val", color="#4CAF50", edgecolor="white")
b2 = ax.bar([i + w/2 for i in x], t95, w, label="Test", color="#2196F3", edgecolor="white")
ax.set_title("mAP@50-95", fontweight="bold")
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.legend(); ax.set_ylim(0, 1)
ax.bar_label(b1, fmt="%.3f", fontsize=7, padding=2)
ax.bar_label(b2, fmt="%.3f", fontsize=7, padding=2)

# ---- Training Time ----
ax = axes[1, 0]
times = [r["train_time_min"] for r in results]
bars = ax.bar(x, times, color=colors[:len(results)], edgecolor="white")
ax.set_title("Training Time", fontweight="bold")
ax.set_ylabel("Minutes")
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.bar_label(bars, fmt="%.0f", fontsize=9, padding=2)

# ---- Inference Speed ----
ax = axes[1, 1]
speeds = [r["inference_ms"] for r in results]
bars = ax.bar(x, speeds, color=colors[:len(results)], edgecolor="white")
ax.set_title("Inference Speed", fontweight="bold")
ax.set_ylabel("ms / image")
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.bar_label(bars, fmt="%.1f", fontsize=9, padding=2)

plt.tight_layout()
plt.savefig("runs/study/comparison_charts.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to runs/study/comparison_charts.png")

In [ ]:
# Efficiency analysis: Accuracy vs Model Size vs Speed
import matplotlib.pyplot as plt
import json
from pathlib import Path

results = json.loads(Path("runs/study/results_summary.json").read_text())
labels = [r["model"].replace("yolo26", "").upper() for r in results]
colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#DDA0DD"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("YOLO26 Efficiency Analysis", fontsize=14, fontweight="bold")

for i, r in enumerate(results):
    ax1.scatter(r["params_M"], r["test_mAP50_95"], s=180, c=colors[i],
               edgecolors="black", zorder=3)
    ax1.annotate(labels[i], (r["params_M"], r["test_mAP50_95"]),
                textcoords="offset points", xytext=(10, 5), fontsize=11, fontweight="bold")
ax1.set_xlabel("Parameters (M)", fontsize=12)
ax1.set_ylabel("Test mAP@50-95", fontsize=12)
ax1.set_title("Accuracy vs Model Size")
ax1.grid(True, alpha=0.3)

for i, r in enumerate(results):
    ax2.scatter(r["inference_ms"], r["test_mAP50_95"], s=180, c=colors[i],
               edgecolors="black", zorder=3)
    ax2.annotate(labels[i], (r["inference_ms"], r["test_mAP50_95"]),
                textcoords="offset points", xytext=(10, 5), fontsize=11, fontweight="bold")
ax2.set_xlabel("Inference Time (ms/image)", fontsize=12)
ax2.set_ylabel("Test mAP@50-95", fontsize=12)
ax2.set_title("Accuracy vs Speed")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("runs/study/efficiency_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to runs/study/efficiency_plots.png")

In [ ]:
# Per-model training curves
from IPython.display import Image, display
from pathlib import Path

study_dir = Path("runs/study")
names = ["yolo26n", "yolo26s", "yolo26m", "yolo26l", "yolo26x"]

for name in names:
    results_png = study_dir / name / "results.png"
    if results_png.exists():
        print(f"\n{'\u2500'*50}")
        print(f"{name.replace('yolo26', 'YOLO26-').upper()} Training Curves")
        print(f"{'\u2500'*50}")
        display(Image(filename=str(results_png), width=900))
    else:
        print(f"{name}: results.png not found")

In [ ]:
# Save everything to Google Drive (optional)
import shutil
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
save_dir = Path("/content/drive/MyDrive/beamfinder_study")
save_dir.mkdir(parents=True, exist_ok=True)

# Copy results JSON + charts
for f in ["results_summary.json", "comparison_charts.png", "efficiency_plots.png"]:
    src = Path(f"runs/study/{f}")
    if src.exists():
        shutil.copy2(src, save_dir / f)
        print(f"Saved {f}")

# Copy best weights + training plots for each model
for name in ["yolo26n", "yolo26s", "yolo26m", "yolo26l", "yolo26x"]:
    best = Path(f"runs/study/{name}/weights/best.pt")
    if best.exists():
        shutil.copy2(best, save_dir / f"{name}_best.pt")
        print(f"Saved {name}_best.pt")
    for plot in ["results.png", "confusion_matrix.png", "F1_curve.png", "PR_curve.png"]:
        src = Path(f"runs/study/{name}/{plot}")
        if src.exists():
            shutil.copy2(src, save_dir / f"{name}_{plot}")

print(f"\nAll results saved to {save_dir}")